<a href="https://colab.research.google.com/github/riya-rajbhut/music-ai/blob/main/basic-neural-network/rnn_basic_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!sudo apt install -y fluidsynth

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fluid-soundfont-gm gsettings-desktop-schemas libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libdouble-conversion3 libevdev2
  libfluidsynth3 libgtk-3-0 libgtk-3-bin libgtk-3-common libgudev-1.0-0
  libinput-bin libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5
  librsvg2-common libwacom-bin libwacom-common libwacom9 libxcb-icccm4
  libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 qsynth qt5-gtk-platformtheme
  qttranslations5-l10n session-migration timgm6mb-soundfont
Suggested packages:
  fluid-soundfont-gs gvfs qt5-image-formats-plugins qtwayland5 jackd
The following NEW packages will be installed:
  at-spi2-core fluid-soundfont

In [3]:
!pip install --upgrade pyfluidsynth

In [4]:
!pip install pretty_midi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 63.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.2 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=e34459872a92b7d3eaaad7bd82911033b83b7b598fe73873dc4e6484cbcb33f9
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [5]:
import collections
import datetime
import fluidsynth
import glob
import numpy as np
import pathlib
import pandas as pd
import pretty_midi
import seaborn as sns
import tensorflow as tf

from IPython import display
from matplotlib import pyplot as plt
from typing import Optional

In [6]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

# Sampling rate for audio playback
_SAMPLING_RATE = 16000

In [20]:
import pathlib
import tensorflow as tf

# 1. Download and extract the file, capturing the returned path to the zip file
zip_path = tf.keras.utils.get_file(
    'maestro-v3.0.0-midi.zip',
    origin='https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip',
    extract=True,
    cache_dir='.', cache_subdir='data',
)

# 2. Point to the actual extracted folder inside 'data/'
# Keras extracts the archive contents directly into 'data/', which contains the 'maestro-v3.0.0' folder




In [35]:
#!dir /s data
#!ls -R data/
import pathlib

# 1. Update this to the actual folder path shown in your file listing
data_dir = pathlib.Path('data/maestro-v2_extracted/maestro-v3.0.0')

# 2. Fix the glob pattern to look recursively (**/) through the subfolders
filenames = list(data_dir.glob('**/*.mid*'))

print('Number of files found:', len(filenames))

Number of files found: 1276


# Process a MIDI file
First, use pretty_midi to parse a single MIDI file and inspect the format of the notes.

In [36]:
sample_file = filenames[1]
print(sample_file)

data/maestro-v2_extracted/maestro-v3.0.0/2018/MIDI-Unprocessed_Schubert7-9_MID--AUDIO_11_R2_2018_wav.midi


In [37]:
#Generate a PrettyMIDI object for the sample MIDI file.
pm = pretty_midi.PrettyMIDI(sample_file)

In [39]:
def display_audio(pm: pretty_midi.PrettyMIDI, seconds=30):
  waveform = pm.fluidsynth(fs=_SAMPLING_RATE)
  # Take a sample of the generated waveform to mitigate kernel resets
  waveform_short = waveform[:seconds*_SAMPLING_RATE]
  return display.Audio(waveform_short, rate=_SAMPLING_RATE)